In [1]:
# Import modules
import os
import random
import torch
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2
import numpy as np
from scipy import ndimage
from PIL import Image, UnidentifiedImageError
import warnings

In [3]:
# import modules
# Initialize SAM2
checkpoint = "/Users/udiyamanshukla_1/Desktop/C_Drive/MSc_project/sam2/checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint, device="cpu"))

# Configuration
consolidated_csv = 'consolidated_coordinates.csv'  # Changed to single CSV file
image_dirs = {'blink': 'Blink_tester', 'open': 'Open_tester'}
output_csv = "eye_segmentation_results_contour_analysis_complete_with_5_dummy.csv"
error_csv = "processing_errors_with_5_dummy.csv"

# Data storage
results = []
error_log = []

# Updated function. Now look at cases where masks > 2 as well
def has_two_non_overlapping_regions(mask, min_size_ratio=0.3): # Was 0.5 earlier, now modified to 0.3. Use 0.3 to 0.45 for best results
    """
    Check if mask has at least two non-overlapping regions where the largest two
    are roughly the same size.
    
    Args:
        mask (ndarray): Binary mask image
        min_size_ratio (float): Minimum ratio between smaller and larger region (default: 0.5)
        
    Returns:
        bool: True if the two largest regions meet the size ratio criteria
    """
    labeled_mask, num_features = ndimage.label(mask)
    
    # Need at least 2 regions to compare
    if num_features < 2:
        return False
    
    # Calculate sizes of all regions
    sizes = [np.sum(labeled_mask == i) for i in range(1, num_features + 1)]
    
    # Get indices of the two largest regions
    largest_two_indices = np.argsort(sizes)[-2:]
    size1, size2 = sizes[largest_two_indices[0]], sizes[largest_two_indices[1]]
    
    # Calculate size ratio (smaller/larger)
    size_ratio = min(size1, size2) / max(size1, size2)
    
    return size_ratio >= min_size_ratio


# def has_two_non_overlapping_regions(mask, min_size_ratio=0.5):
#     """Check if mask has two non-overlapping regions of roughly the same size."""
#     labeled_mask, num_features = ndimage.label(mask)
#     if num_features != 2:
#         return False
#     sizes = [np.sum(labeled_mask == i) for i in range(1, num_features + 1)]
#     size_ratio = min(sizes) / max(sizes)
#     return size_ratio >= min_size_ratio

def calculate_region_aspect_ratio(mask, region_label):
    """Calculate aspect ratio (height/width) of a specific labeled region."""
    region_mask = (mask == region_label)
    contours, _ = cv2.findContours(region_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0
    
    cnt = max(contours, key=cv2.contourArea)
    _, (width, height), _ = cv2.minAreaRect(cnt)
    return height / (width + 1e-6)  # height/width ratio

def is_not_narrow_eyes(mask, aspect_threshold=0.18):
    """
    Check if the two largest regions are not narrow (height/width > threshold).
    Returns False if has_two_non_overlapping_regions is False.
    """
    if not has_two_non_overlapping_regions(mask):
        return False
    
    labeled_mask, num_features = ndimage.label(mask)
    sizes = [np.sum(labeled_mask == i) for i in range(1, num_features + 1)]
    largest_two_indices = np.argsort(sizes)[-2:]
    
    # Calculate aspect ratio for both regions
    aspect1 = calculate_region_aspect_ratio(labeled_mask, largest_two_indices[0]+1)
    aspect2 = calculate_region_aspect_ratio(labeled_mask, largest_two_indices[1]+1)
    
    # Both regions should have height/width > threshold to be considered "not narrow"
    return aspect1 > aspect_threshold and aspect2 > aspect_threshold

def select_best_mask(masks, scores):
    """Select best mask according to new criteria with additional quality checks."""
    top_indices = np.argsort(scores)[-3:][::-1]  # Get top 3 masks by score
    top_masks = [masks[i] for i in top_indices]
    top_scores = [scores[i] for i in top_indices]
    
    # First check: look for masks meeting all criteria
    for i, (mask, score) in enumerate(zip(top_masks, top_scores)):
        mask_area = np.sum(mask)
        if (mask_area < 75000 and 
            has_two_non_overlapping_regions(mask) and 
            is_not_narrow_eyes(mask) and  # New condition
            score >= 0.02):
            return mask, score
    
    # Second check: if no masks meet all criteria, return the highest scoring mask
    best_idx = np.argmax(scores)
    return masks[best_idx], scores[best_idx]


# def select_best_mask(masks, scores):
#     """Select best mask according to new criteria with additional quality checks."""
#     top_indices = np.argsort(scores)[-3:][::-1]  # Get top 3 masks by score
#     top_masks = [masks[i] for i in top_indices]
#     top_scores = [scores[i] for i in top_indices]
    
#     # First check: look for masks meeting all criteria (size, regions, and minimum score)
#     for i, (mask, score) in enumerate(zip(top_masks, top_scores)):
#         mask_area = np.sum(mask)
#         if (mask_area < 75000 and 
#             has_two_non_overlapping_regions(mask) and 
#             score >= 0.052):  # New score threshold for open eye. Can experiemnt with higher thresholds upto 0.2
#             return mask, score
    
    # # Second check: if no masks meet all criteria, return the highest scoring mask
    # best_idx = np.argmax(scores)
    # return masks[best_idx], scores[best_idx]

def log_error(subfolder, img_name, error_type, details=""):
    """Record processing errors to error_log."""
    error_log.append({
        'subfolder': subfolder,
        'image_name': img_name,
        'error_type': error_type,
        'details': str(details)[:200]  # Truncate long error messages
    })
    print(f"! ERROR [{error_type}] {subfolder}/{img_name}: {details}")

def process_images_with_sam2(label, num_images=None):
    """Process images with comprehensive error handling."""
    collected = 0
    
    # Load consolidated CSV once at the beginning
    try:
        coord_df = pd.read_csv(consolidated_csv)
    except Exception as e:
        log_error("GLOBAL", "ALL", "CONSOLIDATED_CSV_LOAD_ERROR", e)
        return

    for subfolder in sorted(os.listdir(image_dirs[label])):
        subfolder_path = os.path.join(image_dirs[label], subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        images = os.listdir(subfolder_path)
        for img_name in images:
            if num_images is not None and collected >= num_images:
                return

            try:
                # Validate CSV entry - now looking in consolidated file
                row = coord_df[coord_df['actual_filename'] == img_name]  # Changed column name
                if row.empty:
                    log_error(subfolder, img_name, "MISSING_CSV_ENTRY")
                    continue

                row = row.iloc[0]
                try:
                    lx, ly = int(row['abs_lx']), int(row['abs_ly'])
                    rx, ry = int(row['abs_rx']), int(row['abs_ry'])
                except (ValueError, KeyError) as e:
                    log_error(subfolder, img_name, "INVALID_COORDINATES", e)
                    continue

                # Load and validate image
                img_path = os.path.join(subfolder_path, img_name)
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("error")
                        image = cv2.imread(img_path)
                        if image is None:
                            raise UnidentifiedImageError("OpenCV cannot read image")
                        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                except (UnidentifiedImageError, Warning, cv2.error) as e:
                    log_error(subfolder, img_name, "IMAGE_LOAD_ERROR", e)
                    continue

                # Process with SAM2
                predictor.set_image(image_rgb)
                masks, scores, logits = predictor.predict(
                    point_coords=np.array([[lx, ly], [rx, ry]]),
                    point_labels=np.array([1, 1]),
                    multimask_output=True
                )

                best_mask, best_score = select_best_mask(masks, scores)
                results.append({
                    'image_name': img_name,
                    'label': label,
                    'lx': lx, 'ly': ly,
                    'rx': rx, 'ry': ry,
                    'mask_area': int(np.sum(best_mask)),
                    'mask_score': float(best_score),
                    'subfolder': subfolder
                })

                collected += 1
                print(f"[{label.upper()}] {subfolder}/{img_name} - Processed")

            except Exception as e:
                log_error(subfolder, img_name, "PROCESSING_ERROR", e)
                continue

# Main execution
if __name__ == "__main__":
    print("Starting processing...")
    process_images_with_sam2('blink',40)
    process_images_with_sam2('open',40)

    # Save successful results
    if results:
        pd.DataFrame(results).to_csv(output_csv, index=False)
        print(f"Saved {len(results)} successful results to {output_csv}")
    else:
        print("No images processed successfully")

    # Save error log
    if error_log:
        pd.DataFrame(error_log).to_csv(error_csv, index=False)
        print(f"Saved {len(error_log)} errors to {error_csv}")
    else:
        print("No errors encountered")

    print("Processing complete")

Starting processing...
[BLINK] 033/2022-10-07 15:08:27.347264-13886.jpg - Processed
[BLINK] 033/2022-10-07 15:12:48.550136-1550.jpg - Processed
[BLINK] 033/2022-10-07 15:13:17.570683-6192.jpg - Processed
[BLINK] 033/2022-10-07 15:25:45.772559-6792.jpg - Processed
[BLINK] 033/2022-10-07 15:08:27.352590-13887.jpg - Processed
[BLINK] 033/2022-10-07 15:31:55.896354-13852.jpg - Processed
[BLINK] 033/2022-10-07 15:08:33.891261-14933.jpg - Processed
[BLINK] 033/2022-10-07 15:13:45.541115-10662.jpg - Processed
[BLINK] 033/2022-10-07 15:12:47.564882-1392.jpg - Processed
[BLINK] 033/2022-10-07 15:13:17.551162-6189.jpg - Processed
[BLINK] 033/2022-10-07 15:31:17.912477-7804.jpg - Processed
[BLINK] 033/2022-10-07 15:12:47.575097-1394.jpg - Processed
[BLINK] 033/2022-10-07 15:31:27.777240-9376.jpg - Processed
[BLINK] 033/2022-10-07 15:08:33.878937-14931.jpg - Processed
[BLINK] 033/2022-10-07 15:08:28.761505-14105.jpg - Processed
[BLINK] 033/2022-10-07 15:25:11.745460-1365.jpg - Processed
[BLINK] 03